# App Clustering Pipeline: Nearest Neighbors + Graph-Based Approach

This notebook groups ~4 000 mobile subscription apps into clusters of semantically similar competitors using a fully unsupervised pipeline:

1. **Sentence embeddings** — encode each app's `features` list with `BAAI/bge-base-en`
2. **Nearest Neighbors** — find the closest apps in embedding space (cosine similarity)
3. **Graph connected components** — connect apps above a similarity threshold; each connected component becomes a cluster
4. **UMAP visualization** — project to 2D for visual inspection

In [ ]:
import pandas as pd
import numpy as np
import umap
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer
import plotly.express as px

## 1. Data Loading & Exploration

Load the raw dataset and inspect its structure.  
Each row is one iOS app with `trackName`, `description`, `overview`, `features`, and screenshot URLs.  
Screenshot URL columns are dropped — they carry no semantic signal for clustering.

In [2]:
df = pd.read_json('subscription_apps.json')
df.head()

,trackName,description,screenshotUrls,ipadScreenshotUrls,appletvScreenshotUrls,overview,features
0,Logo Maker - AI Art Design,Logo Maker AI-Powered Logo Design Made Easy\nC...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],[],Logo Maker is an AI-powered logo design app th...,[AI-Powered Logo Generation with customizable ...
1,Arvin® - AI Logo Maker,Arvin®: Your AI-Powered Logo Maker\n\nTransfor...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],[],Arvin® is an AI-powered logo maker designed fo...,"[AI-generated logo design, Customizable letter..."
2,AI Logo Generator & Design,AI LOGO MAKER: GENERATE STUNNING LOGOS IN SECO...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],AI Logo Generator is a powerful app designed f...,"[AI-powered logo generation in seconds, Over 5..."
3,Logo Maker _ AI Design Creator,Logo Maker allows you to create the logo of yo...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],Logo Maker is a user-friendly app designed for...,"[Access to over 7000 logo templates, Diverse c..."
4,Logo Maker Shop: AI Creator,Logo Maker Shop iOS app lets you create a stun...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],Logo Maker Shop is an intuitive iOS app design...,[10K+ customizable logo templates across 13 ca...


In [3]:
df.describe()

,trackName,description,screenshotUrls,ipadScreenshotUrls,appletvScreenshotUrls,overview,features
count,4264,4264,4264,4264,4264,4264,4264
unique,4262,4263,4264,2411,1,4264,4264
top,TuneIn Radio: Music & Sports,"Listen to All the Audio You Love, All in One P...",[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],[],The G3+ app is a comprehensive platform design...,"[Stream sermons, audiobooks, music, video cour..."
freq,2,2,1,1854,4264,1,1


In [4]:
df.drop(columns=['screenshotUrls', 'ipadScreenshotUrls', 'appletvScreenshotUrls'], inplace=True)

In [5]:
df.head()

,trackName,description,overview,features
0,Logo Maker - AI Art Design,Logo Maker AI-Powered Logo Design Made Easy\nC...,Logo Maker is an AI-powered logo design app th...,[AI-Powered Logo Generation with customizable ...
1,Arvin® - AI Logo Maker,Arvin®: Your AI-Powered Logo Maker\n\nTransfor...,Arvin® is an AI-powered logo maker designed fo...,"[AI-generated logo design, Customizable letter..."
2,AI Logo Generator & Design,AI LOGO MAKER: GENERATE STUNNING LOGOS IN SECO...,AI Logo Generator is a powerful app designed f...,"[AI-powered logo generation in seconds, Over 5..."
3,Logo Maker _ AI Design Creator,Logo Maker allows you to create the logo of yo...,Logo Maker is a user-friendly app designed for...,"[Access to over 7000 logo templates, Diverse c..."
4,Logo Maker Shop: AI Creator,Logo Maker Shop iOS app lets you create a stun...,Logo Maker Shop is an intuitive iOS app design...,[10K+ customizable logo templates across 13 ca...


## 2. Feature Embedding

Encode the `features` column using the `BAAI/bge-base-en` sentence transformer model.

**Why `features` only?**  
Features are the most structured and information-dense column — they directly describe what an app *does*, making them better signal for competitor detection than free-form description text.

**Why L2-normalize?**  
Normalised vectors allow cosine similarity to be computed as a dot product, which is consistent with the `cosine` metric used downstream in KNN.

Embeddings are saved to disk so this step can be skipped on re-runs.

In [ ]:
# Load the pre-trained sentence transformer (768-dim output, optimised for semantic similarity)
model = SentenceTransformer("BAAI/bge-base-en")

# Use the features column — most structured signal for competitor detection
texts = df["features"].fillna("").tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2-normalise so cosine sim = dot product
)

df["embedding_features"] = embeddings.tolist()
df.head()


In [ ]:
np.save("embedding_features.npy", embeddings)

## 3. NN Graph-Based Clustering

**Strategy:** build an undirected similarity graph and extract clusters as connected components — no need to pre-specify the number of clusters.

**Steps:**
1. Fit `NearestNeighbors(k=2, metric='cosine')` — for each app find its 2 closest neighbours
2. Convert distances to cosine similarities (`sim = 1 − dist`)
3. Add a graph edge between two apps if `sim ≥ threshold` (0.8)
4. Extract connected components — each component is one cluster
5. Apps with no edges (too dissimilar to all neighbours) get label `−1` (noise)

In [ ]:
k = 2  # number of nearest neighbours per app (self + 1 true neighbour)
X = np.array(df["embedding_features"].tolist())

# Cosine metric on L2-normalised vectors is equivalent to (2 - 2*dot_product)^0.5
# but sklearn's NearestNeighbors handles this correctly with metric='cosine'
nn = NearestNeighbors(n_neighbors=k, metric='cosine')
nn.fit(X)

# distances: cosine distance (0 = identical, 1 = orthogonal)
# indices:   row indices of the k nearest neighbours for each app
distances, indices = nn.kneighbors(X)


### 3.1 Sanity Check — Inspect Nearest Neighbours

Before building the full graph, visually verify that NN results make sense:  
neighbouring apps should be semantically related (same category, similar features).

In [ ]:
i = 720 

neighbors = indices[i]
dists = distances[i]

for idx, dist in zip(neighbors, dists):
    print(df.iloc[idx]['trackName'], dist)

Bodylura: Fitness & Nutrition 0.0
WeGLOW: Home Workout for Women 0.045196351408324786


In [ ]:
def inspect_neighbors(i, top_k=2):
    print(f"\nAPP: {df.iloc[i]['trackName']}")
    print(df.iloc[i]['overview'])
    print("\n--- NEIGHBORS ---")

    for idx, dist in zip(indices[i][:top_k], distances[i][:top_k]):
        print(f"\n{df.iloc[idx]['trackName']} (dist={dist:.3f})")
        print(df.iloc[idx]['overview'])

In [ ]:
indices.shape, distances.shape

((4264, 2), (4264, 2))

In [12]:
for i in [0, 50, 100]:
    inspect_neighbors(i, top_k=2)


APP: Logo Maker - AI Art Design
Logo Maker is an AI-powered logo design app that enables entrepreneurs, small businesses, and creators to effortlessly create stunning logos in seconds. With no design skills required, users simply describe their vision, and the app generates multiple unique logo designs tailored to their brand.

--- NEIGHBORS ---

Logo Maker - AI Art Design (dist=0.000)
Logo Maker is an AI-powered logo design app that enables entrepreneurs, small businesses, and creators to effortlessly create stunning logos in seconds. With no design skills required, users simply describe their vision, and the app generates multiple unique logo designs tailored to their brand.

AI Logo Generator & Design (dist=0.020)
AI Logo Generator is a powerful app designed for entrepreneurs, creators, and small business owners looking to craft unique logos and graphics effortlessly. With its AI-driven technology, users can generate professional-quality logos in seconds, making it ideal for anyone

### 3.2 Build Similarity Graph & Extract Clusters

Iterate over every app's KNN neighbours. If cosine similarity meets the threshold, add an undirected edge between the two apps.  
Then run connected components — each component is returned as a set of node indices.

In [ ]:
import networkx as nx

G = nx.Graph()

# Minimum cosine similarity to add an edge between two apps.
# 0.8 means the apps must share at least 80% directional alignment in embedding space.
# Lower threshold → larger, looser clusters; higher → smaller, tighter clusters.
threshold = 0.8

for i in range(len(indices)):
    for j, dist in zip(indices[i], distances[i]):
        if i != j:                       # skip self-loop (always distance=0)
            sim = 1 - dist               # convert cosine distance → cosine similarity
            if sim >= threshold:
                G.add_edge(i, j)

# Each connected component is a candidate cluster
components = list(nx.connected_components(G))
print(f"Nodes in graph: {G.number_of_nodes()}  |  Edges: {G.number_of_edges()}  |  Components: {len(components)}")


### 3.3 Assign Cluster Labels

Convert the list of connected components into a flat `cluster` column on the DataFrame.  
Components are numbered sequentially (0, 1, 2 …); isolated apps that never crossed the similarity threshold keep label `−1`.

In [ ]:
# Initialise all apps as noise (-1); only connected apps get a positive cluster id
cluster_labels = np.full(len(df), -1, dtype=int)

for cid, comp in enumerate(components):
    for idx in comp:
        cluster_labels[idx] = cid

df['cluster'] = cluster_labels
print(f"Clustered: {(cluster_labels >= 0).sum()}  |  Noise: {(cluster_labels == -1).sum()}  |  Clusters: {len(components)}")


In [24]:
df.head()

,trackName,description,overview,features,embedding_features,cluster
0,Logo Maker - AI Art Design,Logo Maker AI-Powered Logo Design Made Easy\nC...,Logo Maker is an AI-powered logo design app th...,[AI-Powered Logo Generation with customizable ...,"[0.026286382228136063, 0.014646739698946476, -...",0
1,Arvin® - AI Logo Maker,Arvin®: Your AI-Powered Logo Maker\n\nTransfor...,Arvin® is an AI-powered logo maker designed fo...,"[AI-generated logo design, Customizable letter...","[0.029506200924515724, 0.018569650128483772, 0...",0
2,AI Logo Generator & Design,AI LOGO MAKER: GENERATE STUNNING LOGOS IN SECO...,AI Logo Generator is a powerful app designed f...,"[AI-powered logo generation in seconds, Over 5...","[0.02679363079369068, 0.02018563449382782, -0....",0
3,Logo Maker _ AI Design Creator,Logo Maker allows you to create the logo of yo...,Logo Maker is a user-friendly app designed for...,"[Access to over 7000 logo templates, Diverse c...","[0.01727970503270626, 0.016116071492433548, 0....",1
4,Logo Maker Shop: AI Creator,Logo Maker Shop iOS app lets you create a stun...,Logo Maker Shop is an intuitive iOS app design...,[10K+ customizable logo templates across 13 ca...,"[0.013803043402731419, 0.02349875494837761, 0....",1


In [22]:
df['cluster'].value_counts()

cluster
31     38
261    32
36     29
149    26
287    25
       ..
679     2
682     2
720     2
4       2
3       2
Name: count, Length: 723, dtype: int64

In [29]:
df.to_csv("clustered_apps_NN.csv", index=False)

## 4. UMAP Visualization

Project the 768-dimensional embeddings to 2D using UMAP for visual inspection of cluster structure.  
UMAP preserves local neighbourhood relationships — apps that appear close in the plot were close neighbours in the original embedding space.

The interactive Plotly scatter lets you hover over individual points to see the app name.

In [31]:
umap_model = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2, 
    random_state=42, 
    metric='cosine')
X_umap = umap_model.fit_transform(X)

df['umap_x'] = X_umap[:,0]
df['umap_y'] = X_umap[:,1]

c:\Study\selfeducation\Universe_Group_test_task\.venv\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
fig = px.scatter(
    df,
    x='umap_x',
    y='umap_y',
    color='cluster',               
    hover_data=['trackName'], 
    title="UMAP visualization of app embeddings",
    width=1000,
    height=800,
    color_continuous_scale=px.colors.qualitative.Alphabet
)

fig.show(renderer="browser")

In [242]:
df.iloc[192]

trackName                                       PAW Patrol Rescue World
description           Play as your favorite PAW Patrol™ pup and save...
overview              PAW Patrol™ Rescue World is an interactive adv...
features              [Play as popular PAW Patrol™ characters with u...
final_text            \nApp Name: PAW Patrol Rescue World\nOverview:...
embedding             [-0.03311983123421669, -0.016401901841163635, ...
cluster                                                              12
cluster_test                                                         66
embedding_features    [-0.016508031636476517, -0.026263656094670296,...
Name: 192, dtype: object

In [ ]:
df[df['cluster'] == 2]

,trackName,description,overview,features,final_text,embedding,cluster,cluster_test,embedding_features
6,Canva: AI Photo & Video Editor,Canva is your easy to use photo editor and vid...,Canva is a versatile graphic design app that c...,"[Create and customize social media graphics, E...",\nApp Name: Canva: AI Photo & Video Editor\nOv...,"[0.054937440901994705, 0.014039759524166584, -...",2,2,"[0.038620613515377045, 0.011229723691940308, 0..."
12,DaVinci - Image Generator AI,Transform your words and images into stunning ...,DaVinci AI is a cutting-edge app that transfor...,[AI Art Generator for unique and captivating a...,\nApp Name: DaVinci - Image Generator AI\nOver...,"[0.02350323274731636, 0.017311548814177513, 0....",3,2,"[0.02451218292117119, 0.00041534198680892587, ..."
13,AI Video & Photo Generator,Want to create thrilling AI artwork? Or just l...,AI Video & Photo Generator is an innovative ap...,"[AI Video and Photo Creation, Transform text p...",\nApp Name: AI Video & Photo Generator\nOvervi...,"[0.028834158554673195, -0.0011609565699473023,...",3,2,"[0.03524523973464966, -0.011835707351565361, 0..."
17,IMJ : AI Art Generator,Generate amazing AI photos with IMJ AI Art Gen...,IMJ AI Art Generator is a creative app designe...,[Create unique AI-generated art from text prom...,\nApp Name: IMJ : AI Art Generator\nOverview:\...,"[0.014974107034504414, -0.0027675244491547346,...",3,2,"[0.031308963894844055, -0.010184497572481632, ..."
18,WOMBO Dream - AI Art Generator,Turn words into photos & beautiful digital art...,WOMBO Dream is an AI-powered art generator tha...,[Transform text prompts into unique digital ar...,\nApp Name: WOMBO Dream - AI Art Generator\nOv...,"[0.002545674331486225, -0.018844980746507645, ...",3,2,"[0.018927736207842827, -0.007517669815570116, ..."
...,...,...,...,...,...,...,...,...,...
3688,Brother Artspira,"Artspira - Brother’s mobile embroidery, sewing...",Artspira is a mobile app designed for embroide...,"[Access to 10,000+ designs including 2,000+ Di...",\nApp Name: Brother Artspira\nOverview:\nArtsp...,"[0.010359973646700382, -0.027778316289186478, ...",34,2,"[-0.0007790899253450334, 0.0010918440530076623..."
3874,Voi - AI Avatar Portrait Maker,"Just upload a few photos of yourself, and let ...",Voi is an innovative app that allows users to ...,[AI-powered avatar creation from personal phot...,\nApp Name: Voi - AI Avatar Portrait Maker\nOv...,"[0.03872653469443321, 0.0010401351610198617, 0...",3,2,"[0.05247011035680771, 0.001530470559373498, 0...."
3984,Artimind - AI Video Generator,Relive moments with your loved ones who’ve pas...,Artimind - AI Video Generator is an innovative...,[Powerful photo to video generator with family...,\nApp Name: Artimind - AI Video Generator\nOve...,"[-0.0021656027529388666, -0.001028021331876516...",3,2,"[0.017035871744155884, -0.01942460611462593, -..."
4190,Gram AI: Art Creator,Gram AI - allows you to create over 100 AI pho...,Gram AI is an innovative app that enables user...,[Generate personalized avatars from user photo...,\nApp Name: Gram AI: Art Creator\nOverview:\nG...,"[0.02400834485888481, -0.024643337354063988, 0...",3,2,"[0.03947242349386215, 0.009958050213754177, 0...."
